# Read All Subject-Level Gammatone-8 TRF Results

This notebook reads one cached `gammatone-8` TRF result per subject and creates a tidy table with one row per participant. It does **not** refit any TRFs or modify raw EEG data.

The default `EPOCH` is `chapter-1` because that is currently the complete 49-subject batch. Do not combine it with the incomplete `story-segments` cache in the same report.

In [1]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
from IPython.display import display
from eelbrain import load


def find_project_root(start=Path.cwd()):
    start = Path(start).resolve()
    for path in (start, *start.parents):
        if (path / 'analysis' / 'trf_pipeline').is_dir() and (path / 'data' / 'derived').is_dir():
            return path
    raise FileNotFoundError(f'Could not find the project root from {start}')


PROJECT_ROOT = find_project_root()
SRC_DIR = PROJECT_ROOT / 'src'
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from alice_comprehension.paths import DEFAULT_BIDS_ROOT

print(f'Project root: {PROJECT_ROOT}')
print(f'BIDS root: {DEFAULT_BIDS_ROOT}')

Project root: /Users/yanyuwoo/Desktop/mcmaster-project/alice-comprehension-neural-prediction
BIDS root: /Users/yanyuwoo/Data/bids


## Configuration

Change `EPOCH` only when a complete, internally consistent replacement batch exists. Known audio-alignment failures are recorded independently of the TRF score so that low or negative `r_mean` values are not used as exclusion criteria.

In [2]:
MODEL = 'gammatone-8'
RAW = '0.5-20'
EPOCH = 'chapter-1'
EXPECTED_EEG_CHANNELS = 60
EXPECTED_SUBJECTS = [f'sub-{number:02d}' for number in range(1, 50)]
AUDIO_ALIGNMENT_FAILURES = {'sub-09', 'sub-16'}

CACHE_ROOT = DEFAULT_BIDS_ROOT / 'derivatives' / 'eelbrain' / 'cache' / 'trf'
COMPREHENSION_PATH = PROJECT_ROOT / 'data' / 'derived' / 'comprehension_scores_clean.csv'
RESULTS_DIR = PROJECT_ROOT / 'analysis' / 'trf_pipeline' / 'results'
RESULTS_PATH = RESULTS_DIR / f'trf_{MODEL}_subject_results_{EPOCH}.csv'
QC_PATH = RESULTS_DIR / f'trf_{MODEL}_subject_qc_{EPOCH}.csv'

print(f'Model: {MODEL}')
print(f'Raw: {RAW}')
print(f'Epoch/cache version: {EPOCH}')
print(f'Cache root exists: {CACHE_ROOT.exists()}')
print(f'Comprehension table exists: {COMPREHENSION_PATH.exists()}')

Model: gammatone-8
Raw: 0.5-20
Epoch/cache version: chapter-1
Cache root exists: True
Comprehension table exists: True


In [3]:
def model_cache_candidates(subject):
    subject_number = subject.removeprefix('sub-')
    pattern = f'subject-{subject_number}_raw-{RAW}_epoch-{EPOCH}_*.pickle'
    candidates = []
    load_errors = []
    for path in sorted((CACHE_ROOT / subject).glob(pattern)):
        try:
            result = load.unpickle(path)
        except Exception as exc:
            load_errors.append({'subject': subject, 'path': str(path), 'reason': f'pickle_load_error: {exc!r}'})
            continue
        if getattr(result, 'x', None) == MODEL:
            candidates.append((path, result))
    return candidates, load_errors


def summarize_result(subject, path, result):
    r_values = np.asarray(result.r.x, dtype=float).reshape(-1)
    sensor_names = np.asarray(result.r.sensor.names, dtype=str)
    contains_veog = bool(np.any(sensor_names == 'VEOG'))
    h_values = np.asarray(result.h.x, dtype=float)
    det_values = np.asarray(result.proportion_explained.x, dtype=float).reshape(-1)
    best_index = int(np.argmax(r_values))
    minimum_index = int(np.argmin(r_values))
    finite_result = bool(
        np.isfinite(r_values).all()
        and np.isfinite(h_values).all()
        and np.isfinite(det_values).all()
    )

    cache_qc = 'pass'
    cache_reason = ''
    if len(r_values) != EXPECTED_EEG_CHANNELS or contains_veog:
        cache_qc = 'rerun_required'
        reason_parts = []
        if len(r_values) != EXPECTED_EEG_CHANNELS:
            reason_parts.append(f'expected_{EXPECTED_EEG_CHANNELS}_channels_found_{len(r_values)}')
        if contains_veog:
            reason_parts.append('result_contains_VEOG')
        cache_reason = ';'.join(reason_parts)
    elif not finite_result:
        cache_qc = 'fail'
        cache_reason = 'nonfinite_result'

    audio_qc = 'fail' if subject in AUDIO_ALIGNMENT_FAILURES else 'pass'
    exclusion_reasons = []
    if audio_qc == 'fail':
        exclusion_reasons.append('audio_alignment_failed')
    if cache_reason:
        exclusion_reasons.append(cache_reason)

    return {
        'subject': subject,
        'model': MODEL,
        'raw': RAW,
        'epoch': EPOCH,
        'tracking_r_mean': float(np.mean(r_values)),
        'tracking_r_median': float(np.median(r_values)),
        'tracking_r_max': float(np.max(r_values)),
        'tracking_r_min': float(np.min(r_values)),
        'tracking_r_sd': float(np.std(r_values, ddof=1)),
        'positive_channel_count': int(np.sum(r_values > 0)),
        'positive_channel_fraction': float(np.mean(r_values > 0)),
        'best_channel': str(sensor_names[best_index]),
        'minimum_channel': str(sensor_names[minimum_index]),
        'n_result_channels': int(len(r_values)),
        'contains_veog': contains_veog,
        'h_shape': str(tuple(h_values.shape)),
        'det_mean': float(np.mean(det_values)),
        'finite_result': finite_result,
        'audio_alignment_qc': audio_qc,
        'cache_qc': cache_qc,
        'include_trf_primary': audio_qc == 'pass' and cache_qc == 'pass',
        'trf_exclusion_reason': ';'.join(exclusion_reasons),
        'cache_path': str(path),
        'result_summary': repr(result),
    }

## Read and summarize all cached subject results

Exactly one cache matching the selected subject, raw state, epoch, and model is required. Missing or duplicate model caches are written to the QC table rather than selected silently.

In [4]:
rows = []
qc_rows = []

for subject in EXPECTED_SUBJECTS:
    candidates, load_errors = model_cache_candidates(subject)
    qc_rows.extend(load_errors)
    if len(candidates) == 0:
        qc_rows.append({'subject': subject, 'path': '', 'reason': 'missing_model_cache'})
        continue
    if len(candidates) > 1:
        qc_rows.append({
            'subject': subject,
            'path': ';'.join(str(path) for path, _ in candidates),
            'reason': f'duplicate_model_caches:{len(candidates)}',
        })
        continue
    path, result = candidates[0]
    rows.append(summarize_result(subject, path, result))

results = pd.DataFrame(rows).sort_values('subject').reset_index(drop=True)
qc = pd.DataFrame(qc_rows, columns=['subject', 'path', 'reason'])

print(f'Subject results read: {len(results)}/{len(EXPECTED_SUBJECTS)}')
print(f'Cache discovery/load QC rows: {len(qc)}')

Subject results read: 49/49
Cache discovery/load QC rows: 0


## Merge comprehension scores and define analysis cohorts

`include_prediction_primary` requires a valid primary TRF result and a non-missing comprehension score. `include_sensitivity` additionally applies the pre-existing high-noise exclusion flag.

In [5]:
comprehension = pd.read_csv(COMPREHENSION_PATH)
behavior_columns = [
    'participant_id',
    'correct',
    'total',
    'score_prop',
    'high_noise_flag',
    'exclude_high_noise',
    'notes',
]
results = results.merge(
    comprehension[behavior_columns].rename(columns={'participant_id': 'subject'}),
    on='subject',
    how='left',
    validate='one_to_one',
)

results['include_prediction_primary'] = (
    results['include_trf_primary'] & results['score_prop'].notna()
)
results['include_sensitivity'] = (
    results['include_prediction_primary']
    & results['exclude_high_noise'].fillna(False).astype(bool)
)

def prediction_status(row):
    if not row['include_trf_primary']:
        return f"exclude_trf:{row['trf_exclusion_reason']}"
    if pd.isna(row['score_prop']):
        return 'exclude_prediction:missing_comprehension_score'
    if not row['include_sensitivity']:
        return 'include_primary;exclude_high_noise_sensitivity'
    return 'include_primary_and_sensitivity'


results['analysis_status'] = results.apply(prediction_status, axis=1)

## Save tidy result and QC tables

In [6]:
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
results.to_csv(RESULTS_PATH, index=False)
qc.to_csv(QC_PATH, index=False)

print(f'Wrote subject-level results: {RESULTS_PATH}')
print(f'Wrote cache QC table: {QC_PATH}')

Wrote subject-level results: /Users/yanyuwoo/Desktop/mcmaster-project/alice-comprehension-neural-prediction/analysis/trf_pipeline/results/trf_gammatone-8_subject_results_chapter-1.csv
Wrote cache QC table: /Users/yanyuwoo/Desktop/mcmaster-project/alice-comprehension-neural-prediction/analysis/trf_pipeline/results/trf_gammatone-8_subject_qc_chapter-1.csv


## Reporting summary

In [7]:
primary_trf = results.loc[results['include_trf_primary']]
primary_prediction = results.loc[results['include_prediction_primary']]
sensitivity = results.loc[results['include_sensitivity']]

summary = pd.DataFrame({
    'metric': [
        'expected_subjects',
        'results_read',
        'primary_trf_n',
        'primary_prediction_n',
        'sensitivity_n',
        'primary_trf_r_mean',
        'primary_trf_r_median',
        'subjects_with_positive_r_mean',
    ],
    'value': [
        len(EXPECTED_SUBJECTS),
        len(results),
        len(primary_trf),
        len(primary_prediction),
        len(sensitivity),
        primary_trf['tracking_r_mean'].mean(),
        primary_trf['tracking_r_mean'].median(),
        int((primary_trf['tracking_r_mean'] > 0).sum()),
    ],
})
display(summary)

report_columns = [
    'subject',
    'tracking_r_mean',
    'tracking_r_median',
    'tracking_r_max',
    'tracking_r_min',
    'positive_channel_fraction',
    'n_result_channels',
    'score_prop',
    'audio_alignment_qc',
    'cache_qc',
    'analysis_status',
]
display(results[report_columns])

,metric,value
0,expected_subjects,49.000000
1,results_read,49.000000
2,primary_trf_n,46.000000
3,primary_prediction_n,45.000000
4,sensitivity_n,37.000000
5,primary_trf_r_mean,0.028929
6,primary_trf_r_median,0.028511
7,subjects_with_positive_r_mean,41.000000


,subject,tracking_r_mean,tracking_r_median,tracking_r_max,tracking_r_min,positive_channel_fraction,n_result_channels,score_prop,audio_alignment_qc,cache_qc,analysis_status
0,sub-01,0.032680,0.036614,0.092227,-0.044885,0.883333,60,0.750,pass,pass,include_primary_and_sensitivity
1,sub-02,0.052887,0.060313,0.134519,-0.054336,0.816667,60,0.750,pass,pass,include_primary;exclude_high_noise_sensitivity
2,sub-03,0.032321,0.037901,0.070670,-0.031487,0.850000,60,0.750,pass,pass,include_primary_and_sensitivity
3,sub-04,0.007049,0.006878,0.067024,-0.054959,0.566667,60,0.875,pass,pass,include_primary_and_sensitivity
4,sub-05,0.033667,0.034284,0.082921,-0.013479,0.933333,60,0.875,pass,pass,include_primary_and_sensitivity
5,sub-06,0.031949,0.032910,0.092633,-0.046199,0.833333,60,0.625,pass,pass,include_primary_and_sensitivity
6,sub-07,0.041100,0.047055,0.084603,-0.006431,0.950000,60,0.500,pass,pass,include_primary_and_sensitivity
7,sub-08,0.023952,0.026044,0.084041,-0.012810,0.850000,60,0.750,pass,pass,include_primary_and_sensitivity
8,sub-09,0.046789,0.046946,0.091016,-0.004844,0.933333,60,0.250,fail,pass,exclude_trf:audio_alignment_failed
9,sub-10,0.029256,0.030642,0.098272,-0.020500,0.850000,60,0.625,pass,pass,include_primary_and_sensitivity


## Flagged subjects

This view separates independent QC exclusions from low or negative TRF scores. Negative `tracking_r_mean` alone is not an exclusion criterion.

In [8]:
flagged = results.loc[
    (~results['include_trf_primary'])
    | (~results['include_prediction_primary'])
    | (results['high_noise_flag'].fillna(False).astype(bool)),
    report_columns + ['high_noise_flag', 'trf_exclusion_reason', 'notes', 'cache_path'],
]
display(flagged)
display(qc)

,subject,tracking_r_mean,tracking_r_median,tracking_r_max,tracking_r_min,positive_channel_fraction,n_result_channels,score_prop,audio_alignment_qc,cache_qc,analysis_status,high_noise_flag,trf_exclusion_reason,notes,cache_path
1,sub-02,0.052887,0.060313,0.134519,-0.054336,0.816667,60,0.750,pass,pass,include_primary;exclude_high_noise_sensitivity,True,,exclude: noise,/Users/yanyuwoo/Data/bids/derivatives/eelbrain...
8,sub-09,0.046789,0.046946,0.091016,-0.004844,0.933333,60,0.250,fail,pass,exclude_trf:audio_alignment_failed,False,audio_alignment_failed,exclude: low score,/Users/yanyuwoo/Data/bids/derivatives/eelbrain...
15,sub-16,0.061405,0.064149,0.130810,-0.009959,0.933333,60,0.750,fail,pass,exclude_trf:audio_alignment_failed,False,audio_alignment_failed,use,/Users/yanyuwoo/Data/bids/derivatives/eelbrain...
20,sub-21,0.027767,0.028282,0.071805,-0.037170,0.850000,60,NaN,pass,pass,exclude_prediction:missing_comprehension_score,False,,use,/Users/yanyuwoo/Data/bids/derivatives/eelbrain...
27,sub-28,0.049518,0.050952,0.092474,-0.020691,0.950000,60,0.875,pass,pass,include_primary;exclude_high_noise_sensitivity,True,,exclude: noise,/Users/yanyuwoo/Data/bids/derivatives/eelbrain...
28,sub-29,-0.002753,-0.001451,0.034906,-0.055193,0.450000,60,0.750,pass,pass,include_primary;exclude_high_noise_sensitivity,True,,exclude: noise,/Users/yanyuwoo/Data/bids/derivatives/eelbrain...
30,sub-31,-0.020329,-0.022456,0.020271,-0.052287,0.066667,60,0.750,pass,pass,include_primary;exclude_high_noise_sensitivity,True,,exclude: noise,/Users/yanyuwoo/Data/bids/derivatives/eelbrain...
32,sub-33,-0.006565,-0.013512,0.053798,-0.039607,0.316667,60,1.000,pass,pass,include_primary;exclude_high_noise_sensitivity,True,,exclude: noise,/Users/yanyuwoo/Data/bids/derivatives/eelbrain...
40,sub-41,0.027494,0.033779,0.086261,-0.030078,0.836066,61,0.875,pass,rerun_required,exclude_trf:expected_60_channels_found_61;resu...,False,expected_60_channels_found_61;result_contains_...,use,/Users/yanyuwoo/Data/bids/derivatives/eelbrain...
45,sub-46,-0.037622,-0.038094,-0.009722,-0.060440,0.000000,60,0.375,pass,pass,include_primary;exclude_high_noise_sensitivity,True,,exclude: low score & noise,/Users/yanyuwoo/Data/bids/derivatives/eelbrain...


,subject,path,reason
